# Config Search — Standardized Fixed-vs-Cycle Comparison

Set `QUICK_TEST=True` first to verify plumbing (~10 min), then flip to `False` for real numbers.

## Config

In [ ]:
import os, copy, time, json
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import random as _rnd
from collections import defaultdict
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# work_dir
work_dir = os.getcwd()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("work_dir:", work_dir)
print("device  :", DEVICE)

PROC_DIR = os.path.join(work_dir, "data_processed")
FIG_DIR = os.path.join(work_dir, "figures", "config_search_v4")
LOG_DIR = os.path.join(work_dir, "log")
MDL_DIR = os.path.join(work_dir, "model")
for d in [FIG_DIR, LOG_DIR, MDL_DIR]:
    os.makedirs(d, exist_ok=True)
SPLIT_JSON = os.path.join(LOG_DIR, "split_batteries.json")

# Data layout
TIME_COL, V_COL, I_COL, T_COL, SOH_COL = 0, 1, 2, 3, -1
MINS_PER_SAMPLE = 30
HOURS_PER_SAMPLE = MINS_PER_SAMPLE / 60

# Quick Test
QUICK_TEST = False
QUICK_GRAD_STEPS = 2_000
QUICK_TRAIN_B, QUICK_VAL_B, QUICK_TEST_B = 8, 2, 2

# Cycle detection
I_ACTIVE_THRESHOLD = 0.05
MIN_HALFCYCLE_LEN = 2
GAP_FILL = 2

# Cycle windowing
CYCLE_FIXED_LEN = 128
CYCLE_STRIDE = 3
CYCLE_MAX_SPAN_DAYS = 7
CYCLE_MAX_SPAN = int(CYCLE_MAX_SPAN_DAYS * 24 / HOURS_PER_SAMPLE)

# Per-battery normalization
FRESH_FRACTION = 0.20
STD_FLOOR_FRAC = 0.10
CLIP_SIGMA = 8.0

# Split
SEED, TRAIN_RATIO, VAL_RATIO = 42, 0.70, 0.15

# Training
TOTAL_GRAD_STEPS = 25_000
BATCH_SIZE = 128
LR = 1e-3
WEIGHT_DECAY = 1e-4
HUBER_DELTA = 1.5
PATIENCE = 20
GRAD_CLIP = 1.0
SEEDS = [42, 43, 44]

if QUICK_TEST:
    TOTAL_GRAD_STEPS = QUICK_GRAD_STEPS
    SEEDS = [42]
    print(f"QUICK_TEST=True/ subset split, {TOTAL_GRAD_STEPS:,} grad steps, 1 seed.")

# Configs to compare
CONFIGS = {
    "F1000_global": dict(
        windowing="fixed", W=1000, stride=250, n_cycles=None, norm="global"
    ),
    "F1666_global": dict(
        windowing="fixed", W=1666, stride=416, n_cycles=None, norm="global"
    ),
    "F1166_global": dict(
        windowing="fixed", W=1166, stride=291, n_cycles=None, norm="global"
    ),
    "C5_global": dict(
        windowing="cycle", W=None, stride=None, n_cycles=5, norm="global"
    ),
    "C5_perbat": dict(
        windowing="cycle", W=None, stride=None, n_cycles=5, norm="perbat"
    ),
    "C8_global": dict(
        windowing="cycle", W=None, stride=None, n_cycles=8, norm="global"
    ),
}
MODEL_NAME = "CNN_GRU"  # or "GRU" — model classes defined below

print(
    f"{len(CONFIGS)} configs × {len(SEEDS)} seed(s) = {len(CONFIGS) * len(SEEDS)} runs"
)

## Battery split

In [ ]:
def get_battery_id(fname):
    for p in fname.replace("X_", "").replace("y_", "").split("_"):
        if p.startswith("P") and p[1:].isdigit():
            return p
    return fname.split("_")[1]


proc_files = sorted([f for f in os.listdir(PROC_DIR) if f.endswith(".npy")])
battery_files = defaultdict(list)
for f in proc_files:
    battery_files[get_battery_id(f)].append(f)
for b in battery_files:
    battery_files[b].sort()
batteries_all = sorted(battery_files)
print(f"Batteries on disk: {len(batteries_all)}   files: {len(proc_files)}")

if os.path.exists(SPLIT_JSON):
    with open(SPLIT_JSON) as fh:
        split = json.load(fh)
    train_b, val_b, test_b = split["train"], split["val"], split["test"]
    known = set(train_b) | set(val_b) | set(test_b)
    assert known == set(batteries_all), (
        "split_batteries.json does not match batteries on disk — "
        "delete it to regenerate, but know that changes the test set."
    )
    print(f"Loaded persisted split from {SPLIT_JSON}")
else:
    _bats = batteries_all.copy()
    _rnd.seed(SEED)
    _rnd.shuffle(_bats)
    n = len(_bats)
    train_b = _bats[: int(TRAIN_RATIO * n)]
    val_b = _bats[int(TRAIN_RATIO * n) : int((TRAIN_RATIO + VAL_RATIO) * n)]
    test_b = _bats[int((TRAIN_RATIO + VAL_RATIO) * n) :]
    with open(SPLIT_JSON, "w") as fh:
        json.dump(
            {"seed": SEED, "train": train_b, "val": val_b, "test": test_b}, fh, indent=2
        )
    print(f"Created and persisted split → {SPLIT_JSON}")

assert not (
    set(train_b) & set(val_b) or set(train_b) & set(test_b) or set(val_b) & set(test_b)
)

if QUICK_TEST:
    train_b, val_b, test_b = (
        train_b[:QUICK_TRAIN_B],
        val_b[:QUICK_VAL_B],
        test_b[:QUICK_TEST_B],
    )

batteries_used = train_b + val_b + test_b
print(f"Train:{len(train_b)}  val:{len(val_b)}  test:{len(test_b)}")
print(f"test batteries: {test_b}")

## Load raw data + cycle detection

In [ ]:
raw = {
    b: [np.load(os.path.join(PROC_DIR, f)) for f in battery_files[b]]
    for b in batteries_used
}
for b, arrs in raw.items():
    for a in arrs:
        assert a.shape[1] == 5, (
            f"{b}: expected 5 cols [time,V,I,T,SoH], got {a.shape[1]}"
        )
print(
    f"Loaded {sum(len(v) for v in raw.values())} arrays, "
    f"{sum(a.shape[0] for v in raw.values() for a in v):,} samples"
)


def _split_burst_by_sign(current, s, e, min_len):
    sgn = np.sign(current[s:e]).astype(int)
    for j in range(len(sgn)):
        if sgn[j] == 0:
            sgn[j] = sgn[j - 1] if j > 0 else 1
    segs, k = [], 0
    while k < len(sgn):
        s2, v = k, sgn[k]
        while k < len(sgn) and sgn[k] == v:
            k += 1
        segs.append([s2, k, v])
    cleaned = []
    for seg in segs:
        if cleaned and (seg[1] - seg[0]) < min_len:
            cleaned[-1][1] = seg[1]
        elif cleaned and seg[2] == cleaned[-1][2]:
            cleaned[-1][1] = seg[1]
        else:
            cleaned.append(seg)
    return [
        (s + a, s + b, "charge" if v > 0 else "discharge")
        for a, b, v in cleaned
        if (b - a) >= min_len
    ]


def segment_cycles(
    x, threshold=I_ACTIVE_THRESHOLD, min_len=MIN_HALFCYCLE_LEN, gap_fill=GAP_FILL
):
    current = x[:, I_COL]
    active = np.abs(current) > threshold
    n = len(active)
    runs, i = [], 0
    while i < n:
        if not active[i]:
            i += 1
            continue
        s = i
        while i < n and active[i]:
            i += 1
        runs.append([s, i])
    merged = []
    for r in runs:
        if merged and r[0] - merged[-1][1] <= gap_fill:
            merged[-1][1] = r[1]
        else:
            merged.append(r)
    cycles = []
    for s, e in merged:
        if e - s < min_len:
            continue
        cycles.extend(_split_burst_by_sign(current, s, e, min_len))
    return cycles


cycles_per_file = {b: [segment_cycles(a) for a in raw[b]] for b in batteries_used}

# Numeric coverage check
print(f"\n{'Battery':<9}{'Half-cycles':>12}{'Coverage':>10}")
for b in batteries_used[:12]:
    n_cyc, cov_in, cov_all = 0, 0, 0
    for a, cycs in zip(raw[b], cycles_per_file[b]):
        n_cyc += len(cycs)
        active = np.abs(a[:, I_COL]) > I_ACTIVE_THRESHOLD
        inside = np.zeros(len(a), bool)
        for s, e, _ in cycs:
            inside[s:e] = True
        cov_in += (active & inside).sum()
        cov_all += active.sum()
    cov = cov_in / max(cov_all, 1)
    print(f"{b:<9}{n_cyc:>12}{cov:>9.1%} {'ok' if cov > 0.9 else 'not ok'}")

## Per-battery normalization

In [ ]:
NEED_PERBAT = any(c["norm"] == "perbat" for c in CONFIGS.values())


def fit_fresh_reference(arrs, fresh_fraction=FRESH_FRACTION):
    total = sum(len(a) for a in arrs)
    n_ref = max(int(total * fresh_fraction), 50)
    chunks, got = [], 0
    for a in arrs:
        take = min(len(a), n_ref - got)
        chunks.append(a[:take])
        got += take
        if got >= n_ref:
            break
    ref = np.concatenate(chunks)
    p1 = ref[:, [V_COL, I_COL, T_COL]].mean(axis=0)
    p2 = ref[:, [V_COL, I_COL, T_COL]].std(axis=0)
    full = np.concatenate(arrs)
    p2_floor = STD_FLOOR_FRAC * (full[:, [V_COL, I_COL, T_COL]].std(axis=0) + 1e-8)
    p2 = np.maximum(p2, p2_floor) + 1e-8
    return p1, p2


def apply_perbat(a, p1, p2):
    out = a.copy()
    out[:, [V_COL, I_COL, T_COL]] = np.clip(
        (a[:, [V_COL, I_COL, T_COL]] - p1) / p2, -CLIP_SIGMA, CLIP_SIGMA
    )
    return out


raw_perbat = None
if NEED_PERBAT:
    raw_perbat = {}
    for b in batteries_used:
        p1, p2 = fit_fresh_reference(raw[b])
        raw_perbat[b] = [apply_perbat(a, p1, p2) for a in raw[b]]
    b0 = batteries_used[0]
    assert np.allclose(raw[b0][0][:, SOH_COL], raw_perbat[b0][0][:, SOH_COL])
    worst = max(
        np.abs(a[:, [V_COL, I_COL, T_COL]]).max()
        for b in batteries_used
        for a in raw_perbat[b]
    )
    assert worst <= CLIP_SIGMA + 0.01
    print(
        f"per-battery norm ready — labels untouched (ok), max |channel| = {worst:.2f}
    )
else:
    print("no perbat configs, skipped")

## Window builders + dataset assembly


In [ ]:
def add_derivs(f):
    dv = np.gradient(f[:, 0])
    di = np.gradient(f[:, 1])
    return np.concatenate([f, dv[:, None], di[:, None]], axis=1)


def windows_fixed(a, W, stride, bid):
    X, y, sp, bids = [], [], [], []
    for i in range(0, len(a) - W + 1, stride):
        w = a[i : i + W]
        X.append(add_derivs(w[:, 1:-1].copy()))
        y.append(w[-1, -1])
        sp.append(W)
        bids.append(bid)
    return X, y, sp, bids


def windows_cycle(
    a, cycles, n_cycles, bid, fixed_len=CYCLE_FIXED_LEN, cycle_stride=CYCLE_STRIDE
):
    X, y, sp, bids = [], [], [], []
    for k in range(0, len(cycles) - n_cycles + 1, cycle_stride):
        s, e = cycles[k][0], cycles[k + n_cycles - 1][1]
        if (e - s) > CYCLE_MAX_SPAN:
            continue
        seg = a[s:e]
        if len(seg) < 8:
            continue
        idx = np.linspace(0, len(seg) - 1, fixed_len).astype(int)
        w = seg[idx]
        X.append(add_derivs(w[:, 1:-1].copy()))
        y.append(seg[-1, -1])
        sp.append(e - s)
        bids.append(bid)
    return X, y, sp, bids


def build_dataset(cfg):
    src = raw_perbat if cfg["norm"] == "perbat" else raw

    def load(blist):
        Xs, ys, sp, bids = [], [], [], []
        for b in blist:
            for a, cyc in zip(src[b], cycles_per_file[b]):
                if cfg["windowing"] == "fixed":
                    if len(a) < cfg["W"]:
                        continue
                    X, y, s, bd = windows_fixed(a, cfg["W"], cfg["stride"], b)
                else:
                    X, y, s, bd = windows_cycle(a, cyc, cfg["n_cycles"], b)
                Xs += X
                ys += y
                sp += s
                bids += bd
        if not Xs:
            return (
                np.zeros((0, 1, 5), np.float32),
                np.zeros(0, np.float32),
                np.zeros(0),
                np.array([]),
            )
        return (
            np.array(Xs, np.float32),
            np.array(ys, np.float32),
            np.array(sp),
            np.array(bids),
        )

    Xtr, ytr, sp, _ = load(train_b)
    Xva, yva, _, _ = load(val_b)
    Xte, yte, _, bte = load(test_b)
    mu = Xtr.mean(axis=(0, 1), keepdims=True)
    sd = Xtr.std(axis=(0, 1), keepdims=True) + 1e-10
    return dict(
        Xtr=(Xtr - mu) / sd,
        ytr=ytr,
        Xva=(Xva - mu) / sd,
        yva=yva,
        Xte=(Xte - mu) / sd,
        yte=yte,
        span=sp,
        bte=bte,
    )


datasets = {}
print(
    f"{'Config':<15}{'Train':>9}{'Val':>8}{'Test':>8}{'WinLen':>8}{'Hist(d)':>9}  y_tr range"
)
print("-" * 78)
for name, cfg in CONFIGS.items():
    d = build_dataset(cfg)
    if len(d["ytr"]) < 50 or len(d["yva"]) < 10 or len(d["yte"]) < 10:
        print(f"{name:<15}  too few windows — EXCLUDED from the run")
        continue
    datasets[name] = d
    hist = np.median(d["span"]) * HOURS_PER_SAMPLE / 24
    print(
        f"{name:<15}{len(d['ytr']):>9,}{len(d['yva']):>8,}{len(d['yte']):>8,}"
        f"{d['Xtr'].shape[1]:>8}{hist:>9.1f}"
        f"   [{d['ytr'].min():6.2f},{d['ytr'].max():6.2f}]"
    )

# per-test-battery window counts
print("\nTest windows per battery:")
for name, d in datasets.items():
    counts = {b: int((d["bte"] == b).sum()) for b in test_b}
    print(f"  {name:<15}{counts}")

## Models

In [ ]:
GRU_HIDDEN, GRU_LAYERS, GRU_DROPOUT = 64, 2, 0.05
CNN_BASE_CH, CNN_GRU_HIDDEN, CNN_GRU_LAYERS, CNN_DROPOUT = 32, 128, 2, 0.05


class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, x):
        score = self.v(torch.tanh(self.attn(x)))
        weight = torch.softmax(score, dim=1)
        return (weight * x).sum(dim=1)


class GRUModel(nn.Module):
    def __init__(self, input_size=5):
        super().__init__()
        self.gru = nn.GRU(
            input_size,
            GRU_HIDDEN,
            GRU_LAYERS,
            batch_first=True,
            dropout=GRU_DROPOUT if GRU_LAYERS > 1 else 0.0,
        )
        self.attention = Attention(GRU_HIDDEN)
        self.dropout = nn.Dropout(GRU_DROPOUT)
        self.fc = nn.Linear(GRU_HIDDEN, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(self.dropout(self.attention(out))).squeeze(-1)


class CNN_GRU(nn.Module):
    def __init__(self, input_channels=5):
        super().__init__()
        bc = CNN_BASE_CH
        self.cnn = nn.Sequential(
            nn.Conv1d(input_channels, bc, 7, padding=3),
            nn.BatchNorm1d(bc),
            nn.ReLU(),
            nn.Conv1d(bc, bc * 2, 5, padding=2),
            nn.BatchNorm1d(bc * 2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(bc * 2, bc * 2, 3, padding=1),
            nn.BatchNorm1d(bc * 2),
            nn.ReLU(),
        )
        self.gru = nn.GRU(
            bc * 2,
            CNN_GRU_HIDDEN,
            CNN_GRU_LAYERS,
            batch_first=True,
            bidirectional=True,
            dropout=CNN_DROPOUT if CNN_GRU_LAYERS > 1 else 0.0,
        )
        self.attention = Attention(CNN_GRU_HIDDEN * 2)
        self.head = nn.Sequential(
            nn.Dropout(CNN_DROPOUT),
            nn.Linear(CNN_GRU_HIDDEN * 2, CNN_GRU_HIDDEN),
            nn.ReLU(),
            nn.Linear(CNN_GRU_HIDDEN, 1),
        )

    def forward(self, x):
        x = self.cnn(x.permute(0, 2, 1)).permute(0, 2, 1)
        out, _ = self.gru(x)
        return self.head(self.attention(out)).squeeze(-1)


MODELS = {"CNN_GRU": CNN_GRU, "GRU": GRUModel}
model_class = MODELS[MODEL_NAME]
print(f"{MODEL_NAME} params: {sum(p.numel() for p in model_class().parameters()):,}")

## Seeded training


In [ ]:
def set_seed(s):
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    np.random.seed(s)
    _rnd.seed(s)


def make_loader(X, y, seed=None, shuffle=True):
    g = None
    if shuffle and seed is not None:
        g = torch.Generator()
        g.manual_seed(seed)
    return DataLoader(
        TensorDataset(torch.tensor(X), torch.tensor(y)),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=g,
    )


def train_one(name, d, seed, verbose=True):
    set_seed(seed)
    ytr = d["ytr"]
    spe = max(1, int(np.ceil(len(ytr) / BATCH_SIZE)))
    n_epochs = max(1, int(np.ceil(TOTAL_GRAD_STEPS / spe)))

    train_loader = make_loader(d["Xtr"], ytr, seed=seed)
    val_loader = make_loader(d["Xva"], d["yva"], shuffle=False)
    test_loader = make_loader(d["Xte"], d["yte"], shuffle=False)

    model = model_class(d["Xtr"].shape[2]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    crit = nn.HuberLoss(delta=HUBER_DELTA)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)

    best_val, best_epoch, patience = float("inf"), 0, 0
    best_state = copy.deepcopy(model.state_dict())
    nan_hit = False
    t0 = time.time()

    for epoch in range(1, n_epochs + 1):
        model.train()
        bl = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            bl.append(loss.item())
        train_loss = float(np.mean(bl))

        model.eval()
        vl = []
        with torch.no_grad():
            for xb, yb in val_loader:
                vl.append(crit(model(xb.to(DEVICE)), yb.to(DEVICE)).item())
        val_loss = float(np.mean(vl))
        sched.step()

        if not (np.isfinite(train_loss) and np.isfinite(val_loss)):
            nan_hit = True
            if verbose:
                print(
                    f"    NaN at epoch {epoch} — stopping, keeping best checkpoint (ep {best_epoch})"
                )
            break

        if val_loss < best_val:
            best_val, best_epoch, patience = val_loss, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience += 1
        if patience >= PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in test_loader:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())
    y_pred = np.concatenate(preds)
    y_true = d["yte"]

    per_bat = {}
    for b in np.unique(d["bte"]):
        m = d["bte"] == b
        pb = dict(
            rmse=float(np.sqrt(mean_squared_error(y_true[m], y_pred[m]))),
            n=int(m.sum()),
        )
        pb["r2"] = (
            float(r2_score(y_true[m], y_pred[m])) if m.sum() > 2 else float("nan")
        )
        per_bat[str(b)] = pb

    res = dict(
        name=name,
        seed=seed,
        rmse=float(np.sqrt(mean_squared_error(y_true, y_pred))),
        mae=float(mean_absolute_error(y_true, y_pred)),
        r2=float(r2_score(y_true, y_pred)),
        sigma_ratio=float(y_pred.std() / (y_true.std() + 1e-12)),
        best_epoch=best_epoch,
        epochs_run=epoch,
        n_epochs_budget=n_epochs,
        nan_hit=nan_hit,
        minutes=(time.time() - t0) / 60,
        per_battery=per_bat,
    )
    res["_state"] = best_state
    res["_y_pred"] = y_pred
    if verbose:
        print(
            f"    seed {seed}: R²={res['r2']:.4f}  RMSE={res['rmse']:.3f}  "
            f"σ-ratio={res['sigma_ratio']:.2f}  bestEp={best_epoch}/{epoch}  "
            f"[{res['minutes']:.1f} min]{'NaN' if nan_hit else ''}"
        )
    return res

## Run everything — every config × every seed

In [ ]:
all_runs = {}
t_start = time.time()

for name, d in datasets.items():
    print(f"\n{'─' * 66}\n  {name}   ({MODEL_NAME})\n{'─' * 66}")
    runs = []
    for seed in SEEDS:
        runs.append(train_one(name, d, seed))
    all_runs[name] = runs
    r2s = [r["r2"] for r in runs]
    print(
        f"  → R² = {np.mean(r2s):.4f} ± {np.std(r2s):.4f}   over {len(SEEDS)} seed(s)"
    )

    best = max(runs, key=lambda r: r["r2"])
    torch.save(
        best["_state"],
        os.path.join(
            MDL_DIR, f"cfgsearch_v4_{name}_{MODEL_NAME}_seed{best['seed']}.pt"
        ),
    )

print(f"\nTotal wall time: {(time.time() - t_start) / 60:.1f} min")

## Results

In [ ]:
def agg(runs, key):
    v = np.array([r[key] for r in runs], float)
    return v.mean(), v.std()


hist_days = {
    n: float(np.median(datasets[n]["span"]) * HOURS_PER_SAMPLE / 24) for n in all_runs
}

print("=" * 96)
print(
    f"{'Config':<15}{'Hist(d)':>8}{'R² mean±std':>18}{'RMSE mean±std':>18}"
    f"{'σ-ratio':>10}{'NaN':>5}{'seeds':>7}"
)
print("-" * 96)
order = sorted(all_runs, key=lambda n: -agg(all_runs[n], "r2")[0])
for n in order:
    runs = all_runs[n]
    r2m, r2s = agg(runs, "r2")
    rm, rs = agg(runs, "rmse")
    sm, _ = agg(runs, "sigma_ratio")
    nan = any(r["nan_hit"] for r in runs)
    print(
        f"{n:<15}{hist_days[n]:>8.1f}{r2m:>10.4f} ±{r2s:.4f}"
        f"{rm:>10.3f} ±{rs:.3f}{sm:>10.2f}{'⚠️' if nan else '—':>5}{len(runs):>7}"
    )
print("=" * 96)
print("Read gaps against the ±std columns.")

# per-battery breakdown
best_name = order[0]
print(f"\nPer-battery test RMSE / R² — {best_name} (mean over seeds):")
print(f"{'Battery':<10}{'n_win':>7}{'RMSE':>9}{'R²':>10}")
bats = sorted(all_runs[best_name][0]["per_battery"])
for b in bats:
    rmses = [r["per_battery"][b]["rmse"] for r in all_runs[best_name]]
    r2s = [r["per_battery"][b]["r2"] for r in all_runs[best_name]]
    nwin = all_runs[best_name][0]["per_battery"][b]["n"]
    print(f"{b:<10}{nwin:>7}{np.mean(rmses):>9.3f}{np.nanmean(r2s):>10.3f}")
print(
    "(within-battery R² can be low even when RMSE is small, the SoH range inside "
    "one battery is narrow; RMSE is the deployment-relevant number here)"
)

# JSON export (weights + preds stripped)
export = {
    n: [{k: v for k, v in r.items() if not k.startswith("_")} for r in runs]
    for n, runs in all_runs.items()
}
out_json = os.path.join(LOG_DIR, "config_search_v4_results.json")
with open(out_json, "w") as fh:
    json.dump(
        {
            "quick_test": QUICK_TEST,
            "model": MODEL_NAME,
            "grad_steps": TOTAL_GRAD_STEPS,
            "seeds": SEEDS,
            "test_batteries": test_b,
            "hist_days": hist_days,
            "results": export,
        },
        fh,
        indent=2,
    )
print(f"\nJSON → {out_json}")
if QUICK_TEST:
    print(f" QUICK_TEST run! flip to False for real numbers.")

## Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

names = order
r2m = [agg(all_runs[n], "r2")[0] for n in names]
r2e = [agg(all_runs[n], "r2")[1] for n in names]
colors = ["tab:blue" if n.startswith("F") else "tab:green" for n in names]

axes[0].bar(names, r2m, yerr=r2e, capsize=5, color=colors, alpha=0.85)
axes[0].set_ylabel("Test R² (mean ± std over seeds)")
axes[0].set_title(f"Config comparison — {MODEL_NAME}, {len(SEEDS)} seed(s)")
axes[0].tick_params(axis="x", rotation=30)
axes[0].grid(alpha=0.3, axis="y")

for n in names:
    m, e = agg(all_runs[n], "r2")
    axes[1].errorbar(
        hist_days[n],
        m,
        yerr=e,
        fmt="o",
        ms=9,
        capsize=5,
        color="tab:blue" if n.startswith("F") else "tab:green",
    )
    axes[1].annotate(
        n, (hist_days[n], m), textcoords="offset points", xytext=(7, 4), fontsize=8
    )
axes[1].set_xlabel("Median required history (days)")
axes[1].set_ylabel("Test R²")
axes[1].set_title("Accuracy vs required history")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(
    os.path.join(FIG_DIR, "config_search_v4_summary.png"), dpi=110, bbox_inches="tight"
)
plt.show()